# Greffier, le modele : 02. Donnees harmonisees

But : transformer les actes en un **seul format commun**, quelle que soit la source, et le
publier comme dataset Hugging Face versionne. C'est ce jeu harmonise qu'entraineront les
notebooks suivants (avec, en plus, les actes synthetiques generes a la volee).

Le format d'un record (defini une fois dans `outils.py`, `construit_record`) :

```
{ source, langue, split, image, texte, entites }
```

- `texte` : la transcription brute (sans balise), reference pour le CER / WER.
- `entites` : liste de couples `[type, valeur]` dans notre schema commun a 12 entites.
- `split` : train / valid / test, **fourni par la source**, jamais retire au hasard.

Note d'honnetete sur l'Espagne : la version d'Esposalles annotee en entites (competition IEHHR)
est derriere une inscription au CVC de Barcelone ; la version librement accessible est au niveau
ligne, sans entites. La couverture espagnole passera donc surtout par le **generateur
synthetique** (notebook 03), aligne sur les vrais actes espagnols de la famille. Esposalles
reste un jeu de validation optionnel si on obtient l'acces.

In [ ]:
import os, sys, json, collections
from pathlib import Path

if not (Path('IA-Application') / 'modele' / 'outils.py').exists() and not Path('modele/outils.py').exists():
    os.system('git clone --depth 1 https://github.com/Abdellah427/Greffier.git IA-Application')
for cand in ('IA-Application', '.'):
    if (Path(cand) / 'modele' / 'outils.py').exists():
        sys.path.insert(0, cand)
        break
from modele.outils import (charge_mpopp, extrait_entites_mpopp, texte_propre_mpopp,
                           construit_record, charge_config)
CONFIG = charge_config()
print('Schema cible :', CONFIG['entites_cibles'])

## 1. M-POPP : recuperation
On reprend l'archive du notebook 01 (deja telechargee si la session Colab est la meme).

In [ ]:
RACINE = Path('m-popp')
RACINE.mkdir(exist_ok=True)
ZIP = RACINE / 'm-popp_datasets.zip'
if not ZIP.exists():
    os.system(f'wget -q --show-progress -O "{ZIP}" "{CONFIG["datasets"]["m_popp"]["url"]}"')
if not (RACINE / 'extrait').exists():
    os.system(f'unzip -q -o "{ZIP}" -d "{RACINE / "extrait"}"')
base = RACINE / 'extrait'
gt = charge_mpopp(base, encodage=CONFIG['datasets']['m_popp']['encodage_retenu'])
print({s: len(p) for s, p in gt.items()})

## 2. Index des images
Les cles de label sont des noms de fichier ; on construit une table nom -> chemin une seule
fois (rglob par page serait lent). On note aussi les pages dont l'image manque.

In [ ]:
index_img = {}
for p in base.rglob('*'):
    if p.is_file() and p.suffix.lower() in ('.png', '.jpg', '.jpeg'):
        index_img.setdefault(p.name, p)
print('Images indexees :', len(index_img))
manquantes = [nom for s in gt for nom in gt[s] if nom not in index_img]
print('Pages sans image trouvee :', len(manquantes))

## 3. Construction des records harmonises
Pour chaque page : texte propre + entites dans notre schema + chemin image, via
`construit_record`. On garde le split d'origine. Aucun exemple ecarte, sauf image introuvable.

In [ ]:
records = []
for split, pages in gt.items():
    for nom, donnee in pages.items():
        if nom not in index_img:
            continue
        texte_brut = donnee['text']
        rec = construit_record(
            source='m-popp', langue='fr', split=split,
            image=str(index_img[nom]),
            texte=texte_propre_mpopp(texte_brut),
            entites=extrait_entites_mpopp(texte_brut))
        records.append(rec)
print('Records construits :', len(records))
print('\nExemple (sans le chemin image) :')
ex = {k: v for k, v in records[0].items() if k != 'image'}
print(json.dumps(ex, ensure_ascii=False, indent=1)[:700])

## 4. Controles de coherence
Avant de publier, on verifie : repartition des splits, entites vides, doublons d'images entre
splits (ce serait une fuite). On **affiche** les problemes plutot que de les cacher.

In [ ]:
par_split = collections.Counter(r['split'] for r in records)
sans_entite = [r for r in records if not r['entites']]
print('Repartition :', dict(par_split))
print('Records sans aucune entite :', len(sans_entite))

# Fuite eventuelle : une meme image dans deux splits.
vus = {}
fuites = 0
for r in records:
    cle = Path(r['image']).name
    if cle in vus and vus[cle] != r['split']:
        fuites += 1
    vus[cle] = r['split']
print('Fuites train/test detectees :', fuites)

compte = collections.Counter(t for r in records for t, _ in r['entites'])
print('\nEntites (tout le jeu) :')
for t in CONFIG['entites_cibles']:
    print(f'  {t:16} : {compte.get(t, 0)}')

## 5. Publication sur Hugging Face
On assemble un `DatasetDict` (une split par cle) avec l'image chargee et on pousse vers le Hub.
Le token vient des **secrets Colab** (cle `HF_TOKEN`), jamais du code. C'est le stockage
persistant : les notebooks 04 et 05 rechargeront ce dataset au lieu de refaire tout ceci.

In [ ]:
from datasets import Dataset, DatasetDict, Features, Value, Sequence, Image as HFImage

features = Features({
    'source': Value('string'),
    'langue': Value('string'),
    'split': Value('string'),
    'image': HFImage(),
    'texte': Value('string'),
    'entites': Sequence(Sequence(Value('string'))),
})

def sous_jeu(nom):
    rs = [r for r in records if r['split'] == nom]
    return Dataset.from_list(rs, features=features)

dsd = DatasetDict({
    'train': sous_jeu('train'),
    'validation': sous_jeu('valid'),
    'test': sous_jeu('test'),
})
print(dsd)

In [ ]:
# Token depuis les secrets Colab, puis push. A ne lancer qu'une fois pret.
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get('HF_TOKEN'))
except Exception:
    login()  # hors Colab : saisie interactive

repo = CONFIG['huggingface']['repo_dataset']
dsd.push_to_hub(repo, private=False)
print('Pousse vers https://huggingface.co/datasets/' + repo)

## Ce que ce notebook produit
- Un dataset Hugging Face `Abdellah427/greffier-actes-harmonise` avec image + texte + entites,
  meme schema pour toutes les sources a venir.
- Les controles de fuite et les comptes d'entites, affiches (pas caches).

Suite : notebook 03, le **generateur d'actes synthetiques** (FR + ES) qui emet exactement ce
meme format de record, pour entrainer a grande echelle avant l'affinage sur le reel.